In [ ]:
from datetime import date, datetime, timedelta, time

from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from data_engineering.flights.flight_type import get_flights
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from flights_ai_model.validation import flights_df_comparison
from data_engineering.adsbx.readsb import read_adsbx_readsb
from flights.flights_match_adsb import get_matching_icaos_in_flights
import polars as pl

test_date = date(2026,3,1)
df_adsb = read_adsbx_readsb()
df_gold = get_bts_flights_for_day(test_date)
# df_gold = get_sfdps_flights_day(test_date)
# df_gold = get_matching_icaos_in_flights(df_gold, df_adsb)
# df_test = get_adsbx_flights_for_day(test_date)
df_test = get_flights(test_date)
# df_test = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/2026-03-01.parquet')

expr = pl.col("takeoff_time").dt.date() == test_date 
df_gold = df_gold.filter(expr)
df_test = df_test.filter(expr)
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))

6 overlapping flights found for 2026-03-01


/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_15426/3756719591.py:24: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao")))


In [20]:
from flights.flights_comparison import df_flights_comparision, df_flights_comparison_stats
df = df_flights_comparision(df_test, df_gold, datetime.combine(test_date, time.min), datetime.combine(test_date, time.max), compare_airports=False)
stats = df_flights_comparison_stats(df)
print(stats)

{'true_positive': 16979, 'false_positive': 2368, 'false_negative': 443, 'precision': 0.8776037628572906, 'recall': 0.9745723797497418, 'f1': 0.9235497293916071, 'airport_ident_match_pct': 0.9065904941398197}


In [7]:
df = df.select(["icao", "takeoff_time", "landing_time", "takeoff_airport_ident", "landing_airport_ident", "takeoff_airport_ident_df1", "landing_airport_ident_df1", "same_takeoff_airport_ident", "same_landing_airport_ident"])
df = df.filter(pl.col("same_takeoff_airport_ident") == False)
df

icao,takeoff_time,landing_time,takeoff_airport_ident,landing_airport_ident,takeoff_airport_ident_df1,landing_airport_ident_df1,same_takeoff_airport_ident,same_landing_airport_ident
str,datetime[ms],datetime[ms],str,str,str,str,bool,bool
"""046c56""",2026-03-01 13:03:43.485,2026-03-01 16:39:34.208,"""KTPA""","""TBPB""",null,null,false,false
"""046c56""",2026-03-01 18:33:26.113,2026-03-01 22:11:41.741,"""TBPB""","""KTMB""",null,null,false,false
"""046c56""",null,null,null,null,"""TBPB""","""KTMB""",false,false
"""0d0cf3""",2026-03-01 14:42:58.392,2026-03-01 16:43:59.001,"""KBCT""","""MDPC""",null,null,false,false
"""0d0cf3""",2026-03-01 18:32:39.504,2026-03-01 20:59:57.822,"""MDPC""","""KFLL""",null,null,false,false
…,…,…,…,…,…,…,…,…
"""add5e0""",2026-03-01 18:30:28.244,2026-03-01 19:12:54.256,"""KMIC""","""KEAU""",null,null,false,false
"""add5e0""",2026-03-01 20:41:12.767,2026-03-01 21:27:36.299,"""KEAU""","""KMIC""",null,null,false,false
"""add5e0""",null,null,null,null,"""EAU258007""","""KMIC""",false,false


In [8]:
icao = "ad5dcb"
display(df_test.filter(pl.col("icao") == icao))
display(df_gold.filter(pl.col("icao") == icao))
display(df.filter(pl.col("icao") == icao))


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""ad5dcb""","""""","""""",2026-03-01 00:48:25.148,"""KROW""",2026-03-01 02:18:13.895,"""KDEN""",false,false,false,false,"""""","""""","""""",""""""
"""ad5dcb""","""""","""""",2026-03-01 03:28:21.214,"""KDEN""",2026-03-01 04:32:43.929,"""KSLN""",false,false,false,false,"""""","""""","""""",""""""
"""ad5dcb""","""""","""""",2026-03-01 12:29:28.304,"""KSLN""",2026-03-01 13:32:43.767,"""KDEN""",false,false,false,false,"""""","""""","""""",""""""
"""ad5dcb""","""""","""""",2026-03-01 15:01:28.952,"""KDEN""",2026-03-01 15:51:24.809,null,false,false,false,false,"""""","""""","""""",""""""
"""ad5dcb""","""""","""""",2026-03-01 17:12:42.481,"""KLBL""",2026-03-01 18:06:42.652,"""KDEN""",false,false,false,false,"""""","""""","""""",""""""
"""ad5dcb""","""""","""""",2026-03-01 19:10:51.085,"""KDEN""",2026-03-01 19:52:37.484,null,false,false,false,false,"""""","""""","""""",""""""
"""ad5dcb""","""""","""""",2026-03-01 20:57:27.014,null,2026-03-01 21:55:58.333,"""KDEN""",false,false,false,false,"""""","""""","""""",""""""


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""ad5dcb""",null,"""N960SW""",2026-03-01 00:53:00,"""KROW""",2026-03-01 02:18:00,"""KDEN""",false,false,false,false,null,null,null,"""A3"""
"""ad5dcb""",null,"""N960SW""",2026-03-01 03:35:00,"""KDEN""",2026-03-01 04:32:00,"""KSLN""",false,false,false,false,null,null,null,"""A3"""
"""ad5dcb""",null,"""N960SW""",2026-03-01 12:29:00,"""KSLN""",2026-03-01 13:32:00,"""KDEN""",false,false,false,false,null,null,null,"""A3"""
"""ad5dcb""",null,"""N960SW""",2026-03-01 15:08:00,"""KDEN""",2026-03-01 15:53:00,"""KLBL""",false,false,false,false,null,null,null,"""A3"""
"""ad5dcb""",null,"""N960SW""",2026-03-01 17:17:00,"""KLBL""",2026-03-01 18:06:00,"""KDEN""",false,false,false,false,null,null,null,"""A3"""
"""ad5dcb""",null,"""N960SW""",2026-03-01 19:17:00,"""KDEN""",2026-03-01 20:00:00,"""KRKS""",false,false,false,false,null,null,null,"""A3"""
"""ad5dcb""",null,"""N960SW""",2026-03-01 21:12:00,"""KRKS""",2026-03-01 21:55:00,"""KDEN""",false,false,false,false,null,null,null,"""A3"""


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,icao_df1,callsign_df1,registration_df1,takeoff_time_df1,takeoff_airport_ident_df1,landing_time_df1,landing_airport_ident_df1,pia_df1,ladd_df1,military_df1,interesting_df1,aircraft_type_df1,owner_df1,aircraft_description_df1,category_df1,discard_df1,match_status,same_takeoff_airport_ident,same_landing_airport_ident,same_airport_ident
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,bool,str,bool,bool,bool
"""ad5dcb""","""""","""""",2026-03-01 00:48:25.148,"""KROW""",2026-03-01 02:18:13.895,"""KDEN""",false,false,false,false,"""""","""""","""""","""""","""ad5dcb""",null,"""N960SW""",2026-03-01 00:53:00,"""KROW""",2026-03-01 02:18:00,"""KDEN""",false,false,false,false,null,null,null,"""A3""",false,"""both""",true,true,true
"""ad5dcb""","""""","""""",2026-03-01 03:28:21.214,"""KDEN""",2026-03-01 04:32:43.929,"""KSLN""",false,false,false,false,"""""","""""","""""","""""","""ad5dcb""",null,"""N960SW""",2026-03-01 03:35:00,"""KDEN""",2026-03-01 04:32:00,"""KSLN""",false,false,false,false,null,null,null,"""A3""",false,"""both""",true,true,true
"""ad5dcb""","""""","""""",2026-03-01 12:29:28.304,"""KSLN""",2026-03-01 13:32:43.767,"""KDEN""",false,false,false,false,"""""","""""","""""","""""","""ad5dcb""",null,"""N960SW""",2026-03-01 12:29:00,"""KSLN""",2026-03-01 13:32:00,"""KDEN""",false,false,false,false,null,null,null,"""A3""",false,"""both""",true,true,true
"""ad5dcb""","""""","""""",2026-03-01 15:01:28.952,"""KDEN""",2026-03-01 15:51:24.809,null,false,false,false,false,"""""","""""","""""","""""","""ad5dcb""",null,"""N960SW""",2026-03-01 15:08:00,"""KDEN""",2026-03-01 15:53:00,"""KLBL""",false,false,false,false,null,null,null,"""A3""",false,"""both""",true,false,false
"""ad5dcb""","""""","""""",2026-03-01 17:12:42.481,"""KLBL""",2026-03-01 18:06:42.652,"""KDEN""",false,false,false,false,"""""","""""","""""","""""","""ad5dcb""",null,"""N960SW""",2026-03-01 17:17:00,"""KLBL""",2026-03-01 18:06:00,"""KDEN""",false,false,false,false,null,null,null,"""A3""",false,"""both""",true,true,true
"""ad5dcb""","""""","""""",2026-03-01 19:10:51.085,"""KDEN""",2026-03-01 19:52:37.484,null,false,false,false,false,"""""","""""","""""","""""","""ad5dcb""",null,"""N960SW""",2026-03-01 19:17:00,"""KDEN""",2026-03-01 20:00:00,"""KRKS""",false,false,false,false,null,null,null,"""A3""",false,"""both""",true,false,false
"""ad5dcb""","""""","""""",2026-03-01 20:57:27.014,null,2026-03-01 21:55:58.333,"""KDEN""",false,false,false,false,"""""","""""","""""","""""","""ad5dcb""",null,"""N960SW""",2026-03-01 21:12:00,"""KRKS""",2026-03-01 21:55:00,"""KDEN""",false,false,false,false,null,null,null,"""A3""",false,"""both""",false,true,false


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""add5e0""","""""","""""",2026-03-01 18:30:28.244,"""KMIC""",2026-03-01 19:12:54.256,"""KEAU""",false,false,false,false,"""""","""""","""""",""""""
"""add5e0""","""""","""""",2026-03-01 20:41:12.767,"""KEAU""",2026-03-01 21:27:36.299,"""KMIC""",false,false,false,false,"""""","""""","""""",""""""


In [13]:
df.filter(pl.col("icao") == "add1ff")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,icao_df1,callsign_df1,registration_df1,takeoff_time_df1,takeoff_airport_ident_df1,landing_time_df1,landing_airport_ident_df1,pia_df1,ladd_df1,military_df1,interesting_df1,aircraft_type_df1,owner_df1,aircraft_description_df1,category_df1,discard_df1,match_status
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,bool,str
"""add1ff""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""add1ff""","""N99TM""","""N99TM""",2026-03-01 15:44:00,"""KCNO""",2026-03-01 16:25:00,"""KSDM""",false,false,false,false,"""C210""","""MAGDABENJA HOLDINGS LLC""","""CESSNA 201 Centurion""","""A1""",false,"""df1_only"""
"""add1ff""","""""","""""",2026-03-01 15:44:12.220,null,2026-03-01 17:26:43.768,null,false,false,false,false,"""""","""""","""""","""""","""add1ff""","""N99TM""","""N99TM""",2026-03-01 16:33:00,"""KSDM""",2026-03-01 17:26:00,"""KCNO""",false,false,false,false,"""C210""","""MAGDABENJA HOLDINGS LLC""","""CESSNA 201 Centurion""","""A1""",false,"""both"""


In [3]:
df.group_by("match_status").agg(pl.count())

/var/folders/vc/71y607gj39b8jtlhqbbmxv000000gn/T/ipykernel_60218/1002413364.py:1: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  df.group_by("match_status").agg(pl.count())


match_status,count
str,u32
"""df1_only""",202
"""df0_only""",4
"""both""",6741


In [8]:
display(1.0 - 202/len(df))
display(1.0- 4/len(df))

0.9709227004462357

0.9994242118900245

In [16]:
df_gold

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a008c5""",null,"""N101DQ""",2026-03-01 01:02:00,"""KATL""",2026-03-01 02:20:00,"""KDCA""",false,false,false,false,null,null,null,"""A3"""
"""a008c5""",null,"""N101DQ""",2026-03-01 14:21:00,"""KDCA""",2026-03-01 15:38:00,"""KATL""",false,false,false,false,null,null,null,"""A3"""
"""a008c5""",null,"""N101DQ""",2026-03-01 17:22:00,"""KATL""",2026-03-01 20:15:00,"""KDEN""",false,false,false,false,null,null,null,"""A3"""
"""a008c9""",null,"""N101DU""",2026-03-01 00:01:00,"""KRDU""",2026-03-01 02:32:00,"""KMSP""",false,false,false,false,null,null,null,"""A3"""
"""a008c9""",null,"""N101DU""",2026-03-01 16:44:00,"""KMSP""",2026-03-01 18:50:00,"""KCHS""",false,false,false,false,null,null,null,"""A3"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""adf128""",null,"""N998JE""",2026-03-01 05:55:00,"""KPHX""",2026-03-01 10:09:00,"""KBOS""",false,false,false,false,null,null,null,"""A3"""
"""adf194""",null,"""N998NN""",2026-03-01 01:02:00,"""KMIA""",2026-03-01 03:04:00,"""TJSJ""",false,false,false,false,null,null,null,"""A3"""
"""adf194""",null,"""N998NN""",2026-03-01 11:08:00,"""TJSJ""",2026-03-01 13:41:00,"""KMIA""",false,false,false,false,null,null,null,"""A3"""


In [17]:
13707 / len(df.filter(pl.col("takeoff_time_1").is_not_null()))

0.8914542143600416

In [4]:
df.filter(pl.col("match_status") == False)

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,match,callsign_1,registration_1,takeoff_time_1,takeoff_airport_ident_1,landing_time_1,landing_airport_ident_1,pia_1,ladd_1,military_1,interesting_1,aircraft_type_1,owner_1,aircraft_description_1,category_1,match_1,match_status
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,i8,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,i8,bool
"""a008c5""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""""","""""",2026-03-01 21:43:35.135,"""KDEN""",2026-03-01 23:59:43.995,null,false,false,false,false,"""""","""""","""""","""""",0,false
"""a011ac""",null,"""N103US""",2026-03-01 21:54:00,"""KPHL""",2026-03-01 22:41:00,"""KBOS""",false,false,false,false,null,null,null,"""A3""",0,"""""","""""",2026-03-01 13:52:12.921,"""KPHL""",2026-03-01 16:07:24.395,null,false,false,false,false,"""""","""""","""""","""""",0,false
"""a011ac""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""""","""""",2026-03-01 17:52:09.715,"""MYNN""",2026-03-01 20:09:12.347,"""KPHL""",false,false,false,false,"""""","""""","""""","""""",0,false
"""a011ac""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""""","""""",2026-03-01 21:53:55.926,"""KPHL""",2026-03-01 22:41:28.600,"""KBOS""",false,false,false,false,"""""","""""","""""","""""",0,false
"""a02cf1""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""""","""""",2026-03-01 22:54:43.847,"""KLGA""",2026-03-01 23:59:43.523,null,false,false,false,false,"""""","""""","""""","""""",0,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""adf068""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""""","""""",2026-03-01 16:42:58.478,"""KDFW""",2026-03-01 18:52:13.432,"""MMUN""",false,false,false,false,"""""","""""","""""","""""",0,false
"""adf068""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""""","""""",2026-03-01 20:36:57.231,"""MMUN""",2026-03-01 22:58:43.859,"""KDFW""",false,false,false,false,"""""","""""","""""","""""",0,false
"""adf128""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""""","""""",2026-03-01 14:33:28.182,"""KBOS""",2026-03-01 18:56:56.507,"""TNCA""",false,false,false,false,"""""","""""","""""","""""",0,false


In [5]:
df.filter(pl.col("icao") == "a008c5")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,match,callsign_1,registration_1,takeoff_time_1,takeoff_airport_ident_1,landing_time_1,landing_airport_ident_1,pia_1,ladd_1,military_1,interesting_1,aircraft_type_1,owner_1,aircraft_description_1,category_1,match_1,match_status
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,i8,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,i8,bool
"""a008c5""",null,"""N101DQ""",2026-03-01 01:02:00,"""KATL""",2026-03-01 02:20:00,"""KDCA""",false,false,false,false,null,null,null,"""A3""",1,"""""","""""",2026-03-01 01:02:28.491,"""KATL""",2026-03-01 02:20:28.064,"""KDCA""",false,false,false,false,"""""","""""","""""","""""",1,true
"""a008c5""",null,"""N101DQ""",2026-03-01 14:21:00,"""KDCA""",2026-03-01 15:38:00,"""KATL""",false,false,false,false,null,null,null,"""A3""",1,"""""","""""",2026-03-01 14:21:12.553,"""KDCA""",2026-03-01 15:40:25.995,"""KATL""",false,false,false,false,"""""","""""","""""","""""",1,true
"""a008c5""",null,"""N101DQ""",2026-03-01 17:22:00,"""KATL""",2026-03-01 20:15:00,"""KDEN""",false,false,false,false,null,null,null,"""A3""",1,"""""","""""",2026-03-01 17:22:12.641,"""KATL""",2026-03-01 20:16:12.418,"""KDEN""",false,false,false,false,"""""","""""","""""","""""",1,true
"""a008c5""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""""","""""",2026-03-01 21:43:35.135,"""KDEN""",2026-03-01 23:59:43.995,null,false,false,false,false,"""""","""""","""""","""""",0,false


In [3]:
from datetime import date, datetime, timedelta, time
from data_engineering.flights.flight_type import flights_df_to_flights
from flights.flights_comparison import Match, flights_comparison


def compare_flight_dfs(df_test: pl.DataFrame, df_gold: pl.DataFrame, test_date: date) -> pl.DataFrame:
    start_dt = datetime.combine(test_date, time.min)
    end_dt = datetime.combine(test_date + timedelta(days=1), time.min)

    test_cols = df_test.columns
    gold_cols = df_gold.columns

    rows = []
    icaos = sorted(
        set(df_test.get_column("icao").unique().to_list())
        | set(df_gold.get_column("icao").unique().to_list())
    )

    def empty_test_values():
        return {col: None for col in test_cols}

    def empty_gold_values():
        return {f"{col}_gold": None for col in gold_cols}

    def test_values(row):
        return {col: row[col] for col in test_cols}

    def gold_values(row):
        return {f"{col}_gold": row[col] for col in gold_cols}

    def append_row(test_row, gold_row, test_status, gold_status, match_status):
        row = {}
        row.update(test_values(test_row) if test_row is not None else empty_test_values())
        row.update(gold_values(gold_row) if gold_row is not None else empty_gold_values())
        row["test_match_status"] = test_status.name.lower() if test_status is not None else None
        row["gold_match_status"] = gold_status.name.lower() if gold_status is not None else None
        row["match_status"] = match_status
        rows.append(row)

    for icao in icaos:
        df_0 = df_test.filter(pl.col("icao") == icao).sort("takeoff_time")
        df_1 = df_gold.filter(pl.col("icao") == icao).sort("takeoff_time")

        flights_0 = flights_df_to_flights(df_0)
        flights_1 = flights_df_to_flights(df_1)
        match_0, match_1 = flights_comparison(flights_0, flights_1, start_dt=start_dt, end_dt=end_dt)

        test_rows = df_0.iter_rows(named=True)
        gold_rows = df_1.iter_rows(named=True)
        test_rows = list(test_rows)
        gold_rows = list(gold_rows)

        i0 = 0
        i1 = 0
        while i0 < len(test_rows) or i1 < len(gold_rows):
            status_0 = match_0[i0] if i0 < len(match_0) else None
            status_1 = match_1[i1] if i1 < len(match_1) else None

            if status_0 == Match.TRUE and status_1 == Match.TRUE:
                append_row(test_rows[i0], gold_rows[i1], status_0, status_1, "matched")
                i0 += 1
                i1 += 1
            elif status_0 == Match.DISCARD:
                append_row(test_rows[i0], None, status_0, None, "test_discard")
                i0 += 1
            elif status_1 == Match.DISCARD:
                append_row(None, gold_rows[i1], None, status_1, "gold_discard")
                i1 += 1
            elif status_0 == Match.FALSE and status_1 == Match.FALSE:
                append_row(test_rows[i0], gold_rows[i1], status_0, status_1, "unmatched")
                i0 += 1
                i1 += 1
            elif status_0 == Match.FALSE:
                append_row(test_rows[i0], None, status_0, None, "test_only")
                i0 += 1
            elif status_1 == Match.FALSE:
                append_row(None, gold_rows[i1], None, status_1, "gold_only")
                i1 += 1
            elif status_0 is None:
                append_row(None, gold_rows[i1], None, status_1, "gold_only")
                i1 += 1
            elif status_1 is None:
                append_row(test_rows[i0], None, status_0, None, "test_only")
                i0 += 1

    return pl.DataFrame(rows)


df_matches = compare_flight_dfs(df_test, df_gold, test_date)
df_matches


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,icao_gold,callsign_gold,registration_gold,takeoff_time_gold,takeoff_airport_ident_gold,landing_time_gold,landing_airport_ident_gold,pia_gold,ladd_gold,military_gold,interesting_gold,aircraft_type_gold,owner_gold,aircraft_description_gold,category_gold,test_match_status,gold_match_status,match_status
str,str,str,datetime[μs],str,datetime[μs],str,bool,bool,bool,bool,str,str,str,str,str,null,str,datetime[μs],str,datetime[μs],str,bool,bool,bool,bool,null,null,null,str,str,str,str
"""a008c5""","""""","""""",2026-03-01 01:02:28.491,"""KATL""",2026-03-01 02:20:28.064,"""KDCA""",false,false,false,false,"""""","""""","""""","""""","""a008c5""",null,"""N101DQ""",2026-03-01 01:02:00,"""KATL""",2026-03-01 02:20:00,"""KDCA""",false,false,false,false,null,null,null,"""A3""","""true""","""true""","""matched"""
"""a008c5""","""""","""""",2026-03-01 14:21:12.553,"""KDCA""",2026-03-01 15:40:25.995,"""KATL""",false,false,false,false,"""""","""""","""""","""""","""a008c5""",null,"""N101DQ""",2026-03-01 14:21:00,"""KDCA""",2026-03-01 15:38:00,"""KATL""",false,false,false,false,null,null,null,"""A3""","""true""","""true""","""matched"""
"""a008c5""","""""","""""",2026-03-01 17:22:12.641,"""KATL""",2026-03-01 20:16:12.418,"""KDEN""",false,false,false,false,"""""","""""","""""","""""","""a008c5""",null,"""N101DQ""",2026-03-01 17:22:00,"""KATL""",2026-03-01 20:15:00,"""KDEN""",false,false,false,false,null,null,null,"""A3""","""true""","""true""","""matched"""
"""a008c5""","""""","""""",2026-03-01 21:43:35.135,"""KDEN""",2026-03-01 23:59:43.995,null,false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""false""",null,"""test_only"""
"""a008c9""","""""","""""",2026-03-01 00:01:28.945,"""KRDU""",2026-03-01 02:32:58.685,"""KMSP""",false,false,false,false,"""""","""""","""""","""""","""a008c9""",null,"""N101DU""",2026-03-01 00:01:00,"""KRDU""",2026-03-01 02:32:00,"""KMSP""",false,false,false,false,null,null,null,"""A3""","""true""","""true""","""matched"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""adeddd""","""""","""""",2026-03-01 13:30:42.011,"""KMSP""",2026-03-01 15:26:28.699,"""KDFW""",false,false,false,false,"""""","""""","""""","""""","""adeddd""",null,"""N997NN""",2026-03-01 13:30:00,"""KMSP""",2026-03-01 15:26:00,"""KDFW""",false,false,false,false,null,null,null,"""A3""","""true""","""true""","""matched"""
"""adeddd""","""""","""""",2026-03-01 17:35:55.398,"""KDFW""",2026-03-01 20:50:58.280,"""KSJC""",false,false,false,false,"""""","""""","""""","""""","""adeddd""",null,"""N997NN""",2026-03-01 17:36:00,"""KDFW""",2026-03-01 20:50:00,"""KSJC""",false,false,false,false,null,null,null,"""A3""","""true""","""true""","""matched"""
"""adf06d""","""""","""""",2026-03-01 04:06:28.514,"""KATL""",2026-03-01 04:47:58.271,"""KCLT""",false,false,false,false,"""""","""""","""""","""""","""adf06d""",null,"""N998AT""",2026-03-01 04:06:00,"""KATL""",2026-03-01 04:47:00,"""KCLT""",false,false,false,false,null,null,null,"""A3""","""true""","""true""","""matched"""


In [ ]:
df_matches

In [4]:
n_matched = df_matches.filter(pl.col("match_status") == "matched").height

# Discard means an expected day-boundary extra, so exclude it from precision/recall denominators.
n_test = df_matches.filter(pl.col("test_match_status").is_in(["true", "false"])).height
n_gold = df_matches.filter(pl.col("gold_match_status").is_in(["true", "false"])).height

precision = n_matched / n_test if n_test else 0
recall = n_matched / n_gold if n_gold else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

stats = pl.DataFrame({
    "n_matched": [n_matched],
    "n_test": [n_test],
    "n_gold": [n_gold],
    "precision_pct": [precision * 100],
    "recall_pct": [recall * 100],
    "f1_pct": [f1 * 100],
})
stats


n_matched,n_test,n_gold,precision_pct,recall_pct,f1_pct
i64,i64,i64,f64,f64,f64
6886,7087,6950,97.163821,99.079137,98.112132
